# RAD_GENIE_AGENT_API_SMOKE_TEST

Use this companion notebook for Module 03 of the lab.

Before you run the code cell below:

* Attach this notebook to notebook compute that supports Python, such as serverless notebook compute or a cluster.
* Do not attach this notebook to a SQL warehouse.
* Replace `GENIE_AGENT_ID` with the Agent ID from **Configure > About this agent** in `RAD_GENIE_STORE_SALES`.

This notebook is a smoke test and integration check. It verifies that the Genie Agent can be called programmatically, returns `COMPLETED` responses, and exposes inspectable generated SQL and answer payloads.


In [ ]:
from databricks.sdk import WorkspaceClient
import json
import requests
import time

# Replace with your actual Genie Agent ID, found in Configure > About this agent
# or in the agent URL.
GENIE_AGENT_ID = "[INSERT_ID]"

benchmark_questions = [
    "Which Southeast stores had the highest average basket size during the sample week of 2026-07-01 through 2026-07-07?",
    "What was total skincare sales in the Southeast region during the sample week of 2026-07-01 through 2026-07-07?",
    "Which store has the highest conversion rate in the sample data?",
]

expected_answers = {
    benchmark_questions[0]: "Radiance Buckhead (RAD-1001) at 40.60, then Radiance Dilworth (RAD-1005) at 33.10",
    benchmark_questions[1]: "42.00 total skincare sales in the Southeast sample week",
    benchmark_questions[2]: "Radiance SoHo (RAD-1002) with conversion_rate 0.3120",
}

workspace_client = WorkspaceClient()
workspace_host = workspace_client.config.host.rstrip("/")
auth_headers = workspace_client.config.authenticate()

# Product terminology is Genie Agent, but the current REST path still uses /genie/spaces/.
def start_conversation(agent_id: str, question: str) -> dict:
    response = requests.post(
        f"{workspace_host}/api/2.0/genie/spaces/{agent_id}/start-conversation",
        headers=auth_headers,
        json={"content": question},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def get_message(agent_id: str, conversation_id: str, message_id: str) -> dict:
    response = requests.get(
        f"{workspace_host}/api/2.0/genie/spaces/{agent_id}/conversations/{conversation_id}/messages/{message_id}",
        headers=auth_headers,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def wait_for_response(agent_id: str, question: str, poll_seconds: int = 2, timeout_seconds: int = 90) -> dict:
    started = start_conversation(agent_id, question)
    conversation_id = started["conversation"]["id"]
    message_id = started["message"]["id"]

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        message = get_message(agent_id, conversation_id, message_id)
        status = message.get("status")
        if status == "COMPLETED":
            return message
        if status in {"FAILED", "CANCELLED"}:
            raise RuntimeError(json.dumps(message, indent=2))
        time.sleep(poll_seconds)

    raise TimeoutError(f"Timed out waiting for response to: {question}")


for question in benchmark_questions:
    print(f"\nTesting Genie Agent question: {question}")
    print(f"Expected answer: {expected_answers[question]}")
    message = wait_for_response(GENIE_AGENT_ID, question)
    print(f"Status: {message.get('status')}")
    print(json.dumps(message, indent=2))


## How to interpret the smoke test results

### What you just did

You ran a programmatic smoke test against a Genie Agent by sending three benchmark questions through the Genie Agent Conversation API from a Python notebook. For each question, the notebook:

* started a new conversation against the Agent ID
* polled until the response finished
* printed the execution status
* printed the full JSON payload, including the generated SQL and final answer text

### Why this matters

This proves the Genie Agent is not just working interactively in chat, but is also callable through code. That matters because it gives you:

* repeatable validation instead of one-off manual clicking
* inspectable SQL for technical review
* a foundation for regression testing after you change instructions, sources, or examples
* stronger competitive proof that the agent is governed and testable, not just demo-friendly

### What good results look like

A strong result has all of the following:

* `Status: COMPLETED` for every benchmark question
* a JSON payload for every question, not just printed prompts
* a `query.statement_id` in the response, which shows the agent generated executable SQL
* `query_result.row_count` that makes sense for the question
* an answer text that matches the expected business result closely enough to be considered correct

### How to judge this run

For this sample run, the smoke test passed its main objective if all three questions returned `COMPLETED` and the answers matched the sample-data expectations:

* Highest Southeast average basket size: Radiance Buckhead at 40.60
* Southeast skincare sales during the sample week: 42.00
* Highest conversion rate in the sample data: Radiance SoHo at 0.3120

### What to check carefully

Check both correctness and grounding:

* Correctness: does the final text answer match the expected business answer?
* Grounding: does the generated SQL use the right tables, joins, filters, and date window?
* Scope: does the agent answer the asked question, or a slightly different one?
* Consistency: if you re-run the notebook, do you continue to get equivalent answers?

### Important nuance from this run

The first response returned only one top-ranked store in the final answer text, even though the expected answer names both Radiance Buckhead and Radiance Dilworth. The generated SQL uses `RANK()` and filters to `rnk = 1`, so ties should be preserved. That means you should inspect the query result behind the response and confirm whether there was actually a tie or whether the expected answer should be updated.

### When to treat a run as failed

Treat the smoke test as failed if any of these happen:

* any benchmark returns `FAILED`, `CANCELLED`, or times out
* no JSON payload appears
* the SQL clearly uses the wrong tables or wrong date range
* the business answer does not match the known sample-data expectation
* the agent returns a plausible-sounding answer that is not supported by the SQL result
